# Fake Job Detection in Pakistan
## Model Evaluation Notebook

We load the saved `best_model.pkl` and measure Accuracy, Precision, Recall,
F1 Score, and show the Confusion Matrix.

### 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
sns.set(style='whitegrid')


### 2. Settings (must match the training notebook)

In [ ]:
DATASET_PATH = '../data/fake_jobs_dataset.csv'
TARGET_COL   = 'fraudulent'
TEXT_COLS    = ['title', 'company_profile', 'description', 'requirements', 'benefits']


### 3. Load the dataset and the saved model

In [ ]:
df = pd.read_csv(DATASET_PATH)
model = joblib.load('../model/best_model.pkl')
print('Data:', df.shape)
print('Model loaded.')


### 4. Prepare the text and label the same way as training

In [ ]:
import re

def clean_text(text):
    # Clean text the SAME way the backend does.
    text = str(text).lower()
    text = re.sub(r"<[^>]*>", " ", text)     # remove html tags
    text = re.sub(r"[^a-z ]", " ", text)      # keep letters and spaces only
    text = re.sub(r" +", " ", text).strip()   # collapse multiple spaces
    return text


In [ ]:
df = df.dropna(subset=[TARGET_COL])
text_cols = [c for c in TEXT_COLS if c in df.columns]
for c in text_cols:
    df[c] = df[c].fillna('')

df['clean_text'] = df[text_cols].agg(' '.join, axis=1).apply(clean_text)

values = set(df[TARGET_COL].dropna().unique())
if values <= {0, 1}:
    y = df[TARGET_COL].map({1: 'Fake', 0: 'Real'})
else:
    y = df[TARGET_COL].astype(str)
X = df['clean_text']


### 5. Recreate the same test set
Same `random_state=42` so the test set matches training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
y_pred = model.predict(X_test)


### 6. Main scores  (Fake is the positive class)

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label='Fake', zero_division=0)
rec  = recall_score(y_test, y_pred, pos_label='Fake', zero_division=0)
f1   = f1_score(y_test, y_pred, pos_label='Fake', zero_division=0)

print(f'Accuracy : {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall   : {rec:.3f}')
print(f'F1 Score : {f1:.3f}')


### 7. Full classification report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))


### 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Real', 'Fake'])
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix - Best Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


### 9. Summary table

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [round(acc, 3), round(prec, 3), round(rec, 3), round(f1, 3)]
})
summary


Evaluation complete. These scores can go directly into your project report.